# DrawNet — Colab Training

End-to-end pipeline: data download → annotation → caching → training.

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Upload your Kaggle token when prompted in the credentials cell
3. Upload the project zip to Drive OR let the notebook clone from GitHub

**Estimated times on T4:**
- Data download: ~5 min
- Annotation: ~2 min
- Caching 50k images: ~8 min
- Training 50 epochs: ~2 hours

## 1. Mount Google Drive (for persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/DrawNet'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Project will persist at: {DRIVE_DIR}')

## 2. Install dependencies

In [ ]:
!pip install -q kaggle scikit-learn tqdm pyyaml
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE — check Runtime type!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## 3. Set up project

**Option A (recommended):** Upload your project as a zip to Drive, then run the cell below.

**Option B:** Push to GitHub and clone here.

The project zip should contain the `drawnet/` folder with `src/`, `configs/`, `notebooks/`.

In [ ]:
import os, shutil

PROJECT_ZIP = f'{DRIVE_DIR}/drawnet.zip'   # <-- upload drawnet.zip to Drive/DrawNet/
WORK_DIR    = '/content/drawnet'

if os.path.exists(PROJECT_ZIP):
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    !unzip -q "{PROJECT_ZIP}" -d /content/
    print(f'Extracted to {WORK_DIR}')
else:
    # Option B: clone from GitHub
    GITHUB_REPO = 'YOUR_GITHUB_USERNAME/drawnet'   # <-- change this
    !git clone https://github.com/{GITHUB_REPO} {WORK_DIR}

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 4. Kaggle credentials

In [ ]:
import os, json

# Option A: paste your token directly
KAGGLE_USERNAME = 'your_kaggle_username'   # <-- fill in
KAGGLE_KEY      = 'your_kaggle_api_key'    # <-- fill in

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
with open(kaggle_json, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(kaggle_json, 0o600)
print('Kaggle credentials written.')

## 5. Download datasets

Skip any cell if you already have the data in Drive and just want to copy it.

In [ ]:
# ── QuickDraw .npy files ──────────────────────────────────────────────────────
import kaggle, os
kaggle.api.authenticate()

QD_RAW = 'data/raw/quickdraw'
os.makedirs(QD_RAW, exist_ok=True)

CATEGORIES = [
    'face', 'house', 'tree', 'cat', 'car',
    'bird', 'dog', 'fish', 'flower', 'bicycle'
]

# Check Drive cache first
QD_DRIVE = f'{DRIVE_DIR}/data/raw/quickdraw'
if os.path.exists(QD_DRIVE) and len(os.listdir(QD_DRIVE)) >= len(CATEGORIES):
    print('QuickDraw found in Drive — copying...')
    import shutil
    shutil.copytree(QD_DRIVE, QD_RAW, dirs_exist_ok=True)
else:
    print('Downloading QuickDraw from Kaggle...')
    kaggle.api.dataset_download_files(
        'google/quickdraw-dataset',
        path=QD_RAW,
        unzip=True,
        quiet=False
    )
    # Keep only the categories we need
    all_files = os.listdir(QD_RAW)
    keep = {f'{c}.npy' for c in CATEGORIES}
    for f in all_files:
        if f not in keep:
            os.remove(os.path.join(QD_RAW, f))
    # Back up to Drive
    os.makedirs(QD_DRIVE, exist_ok=True)
    shutil.copytree(QD_RAW, QD_DRIVE, dirs_exist_ok=True)
    print('Saved to Drive.')

print('QuickDraw files:', os.listdir(QD_RAW))

In [ ]:
# ── DAP dataset ───────────────────────────────────────────────────────────────
DAP_RAW   = 'data/raw/dap'
DAP_DRIVE = f'{DRIVE_DIR}/data/raw/dap'
os.makedirs(DAP_RAW, exist_ok=True)

if os.path.exists(DAP_DRIVE) and os.listdir(DAP_DRIVE):
    print('DAP found in Drive — copying...')
    shutil.copytree(DAP_DRIVE, DAP_RAW, dirs_exist_ok=True)
else:
    print('Downloading DAP from Kaggle...')
    kaggle.api.dataset_download_files(
        'mostafaabla/dap-dataset',
        path=DAP_RAW,
        unzip=True,
        quiet=False
    )
    shutil.copytree(DAP_RAW, DAP_DRIVE, dirs_exist_ok=True)
    print('Saved to Drive.')

# Show structure
for root, dirs, files in os.walk(DAP_RAW):
    level = root.replace(DAP_RAW, '').count(os.sep)
    if level < 2:
        print(' ' * level * 2 + os.path.basename(root) + f'/ ({len(files)} files)')

## 6. Run auto-annotator (rule-based deviation labels for DAP)

In [ ]:
ANNOT_DRIVE = f'{DRIVE_DIR}/data/processed/deviation/annotations.csv'
ANNOT_LOCAL = 'data/processed/deviation/annotations.csv'

if os.path.exists(ANNOT_DRIVE):
    print('Annotations found in Drive — copying...')
    os.makedirs(os.path.dirname(ANNOT_LOCAL), exist_ok=True)
    shutil.copy(ANNOT_DRIVE, ANNOT_LOCAL)
else:
    print('Running auto_annotate.py...')
    !python src/auto_annotate.py --dap_dir data/raw/dap --out data/processed/deviation/annotations.csv
    shutil.copy(ANNOT_LOCAL, ANNOT_DRIVE)
    print('Saved to Drive.')

import pandas as pd
df = pd.read_csv(ANNOT_LOCAL)
print(f'Annotations: {len(df)} images')
print(df[['rotation','closure_failure','perseveration','spatial_disorganization','size_distortion','omission']].mean().round(3))

## 7. Cache augmented QuickDraw images (one-time, ~8 min)

In [ ]:
CACHE_DRIVE = f'{DRIVE_DIR}/data/augmented'
CACHE_LOCAL = 'data/augmented'

if os.path.exists(f'{CACHE_DRIVE}/index.csv'):
    print('Cache found in Drive — symlinking...')
    if not os.path.exists(CACHE_LOCAL):
        os.symlink(CACHE_DRIVE, CACHE_LOCAL)
    print(f'Cache has {len(pd.read_csv(CACHE_LOCAL + "/index.csv"))} images.')
else:
    print('Building cache (one-time ~8 min)...')
    !python src/cache_dataset.py \
        --config configs/config_colab.yaml \
        --samples_per_class 5000 \
        --out {CACHE_LOCAL}
    # Back up to Drive
    print('Copying cache to Drive (this may take a few minutes)...')
    shutil.copytree(CACHE_LOCAL, CACHE_DRIVE)
    print('Cache saved to Drive.')

## 8. Train DrawNet

Uses `configs/config_colab.yaml`: pretrained ResNet-50, batch_size=64, num_workers=4.

Checkpoints are saved to `outputs/checkpoints/` and backed up to Drive after training.

In [ ]:
# Check for existing checkpoint to resume from
CKPT_DRIVE = f'{DRIVE_DIR}/outputs/checkpoints'
CKPT_LOCAL = 'outputs/checkpoints'
os.makedirs(CKPT_LOCAL, exist_ok=True)

resume_flag = ''
if os.path.exists(f'{CKPT_DRIVE}/best.pt'):
    shutil.copy(f'{CKPT_DRIVE}/best.pt', f'{CKPT_LOCAL}/best.pt')
    # Find the latest epoch checkpoint
    epoch_ckpts = sorted([
        f for f in os.listdir(CKPT_DRIVE) if f.startswith('epoch_')
    ])
    if epoch_ckpts:
        latest = epoch_ckpts[-1]
        shutil.copy(f'{CKPT_DRIVE}/{latest}', f'{CKPT_LOCAL}/{latest}')
        resume_flag = f'--resume {CKPT_LOCAL}/{latest}'
        print(f'Resuming from {latest}')
    else:
        resume_flag = f'--resume {CKPT_LOCAL}/best.pt'
        print('Resuming from best.pt')
else:
    print('No checkpoint found — training from scratch.')

print(f'Resume flag: {resume_flag or "(none)"}')

In [ ]:
!python -u src/train.py \
    --config configs/config_colab.yaml \
    --device cuda \
    {resume_flag}

## 9. Back up checkpoints + logs to Drive

In [ ]:
import shutil

for folder in ['outputs/checkpoints', 'outputs/logs', 'outputs/results']:
    src = f'/content/drawnet/{folder}'
    dst = f'{DRIVE_DIR}/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Backed up {folder}')

print('Done. Checkpoints and logs saved to Drive.')

## 10. Quick evaluation — view training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('outputs/logs/train_log.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(log['epoch'], log['train_loss'], label='Train')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val')
axes[0].set_title('Total Loss'); axes[0].legend()

axes[1].plot(log['epoch'], log['intent_top1_acc'])
axes[1].set_title('Intent Top-1 Accuracy')
axes[1].axhline(0.1, color='gray', linestyle='--', label='random')
axes[1].legend()

axes[2].plot(log['epoch'], log['dev_macro_f1'])
axes[2].set_title('Deviation Macro F1')

plt.tight_layout()
plt.savefig('outputs/results/training_curves.png', dpi=150)
plt.show()
print(log[['epoch','val_loss','intent_top1_acc','dev_macro_f1']].tail(10).to_string(index=False))